# LUCID — DistilBERT training on Colab

**Run this on [Duke's Colab](https://colab.research.google.com/) with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

This notebook:
1. Clones the LUCID repo
2. Installs dependencies
3. Loads the labeled corpus from `data/processed/labeled.csv` (must already be committed to the branch — run `make label` locally first)
4. Fine-tunes DistilBERT with the composite + 6-dimension multi-output head
5. Evaluates on the held-out test split
6. Pushes the trained model to HuggingFace Hub (optional)

**Expected runtime on T4:** ~20 minutes for 3 epochs on ~4,000 labeled examples.


## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
# Clone the repo — replace BRANCH with the current dev branch name
REPO_URL = "https://github.com/lindsaygross/Lucid.git"
BRANCH = "claude/plan-dl-project-1Reh0"  # update after merging to main

!git clone --branch {BRANCH} {REPO_URL} /content/Lucid
%cd /content/Lucid

In [ ]:
!pip install -q -r requirements-dev.txt

In [ ]:
import sys
sys.path.insert(0, "/content/Lucid")

import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from backend.inference.deep import BASE_MODEL_NAME, LucidDistilBERT
from backend.inference.schemas import DIMENSIONS
from scripts.splits import load_split, make_splits
from scripts.train_deep import LucidDataset, train_one_epoch, eval_composite_mae

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("dimensions:", DIMENSIONS)

## 2. Load labeled data

`data/processed/labeled.csv` must be present on the branch. Run `make label` locally first to populate it, then commit + push.


In [ ]:
from pathlib import Path

LABELED = Path("data/processed/labeled.csv")
CORPUS = Path("data/processed/corpus.csv")
SPLITS = Path("data/processed/splits.json")

assert LABELED.exists(), "Run `make label` locally and push the labeled.csv before training"
assert CORPUS.exists(), "Missing corpus.csv — run scripts/build_corpus.py first"

if not SPLITS.exists():
    make_splits(LABELED, CORPUS, SPLITS, val_size=0.15, test_size=0.15, seed=42)

train, val, test = load_split(LABELED, CORPUS, SPLITS)
print(f"train={len(train)}  val={len(val)}  test={len(test)}")
train.head()

## 3. Distribution check

Confirm the label distribution looks reasonable before training.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, dim in zip(axes.flatten(), DIMENSIONS):
    train[dim].value_counts().sort_index().plot(kind="bar", ax=ax, color="#7c3aed")
    ax.set_title(dim)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 4. Train

In [ ]:
EPOCHS = 4
BATCH_SIZE = 32
LR = 2e-5
MAX_LENGTH = 256
DIM_LOSS_WEIGHT = 1.0
WARMUP_RATIO = 0.1

OUTPUT_DIR = Path("models/distilbert")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
train_ds = LucidDataset(train, tokenizer, max_length=MAX_LENGTH)
val_ds = LucidDataset(val, tokenizer, max_length=MAX_LENGTH)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = LucidDistilBERT(BASE_MODEL_NAME, num_dimensions=len(DIMENSIONS)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_steps),
    num_training_steps=total_steps,
)

best_val = float("inf")
history = []
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device, dim_loss_weight=DIM_LOSS_WEIGHT)
    val_mae = eval_composite_mae(model, val_loader, device)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_mae": val_mae})
    print(f"epoch {epoch + 1}  train_loss={train_loss:.4f}  val_mae={val_mae:.2f}")
    if val_mae < best_val:
        best_val = val_mae
        torch.save(model.state_dict(), OUTPUT_DIR / "pytorch_model.bin")
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ saved best checkpoint (val_mae={val_mae:.2f})")

## 5. Evaluate on test set

In [ ]:
from backend.inference.deep import DeepPredictor
from scripts.eval_common import evaluate_predictor, save_metrics, plot_confusion_matrices

predictor = DeepPredictor(OUTPUT_DIR, device=device)
metrics = evaluate_predictor(predictor, test)
save_metrics(metrics, Path("data/outputs/metrics/deep.json"))
plot_confusion_matrices(metrics, Path("data/outputs/figures/deep"), title_prefix="deep · ")

print(f"macro F1: {metrics['macro_f1']:.3f}")
print(f"macro accuracy: {metrics['macro_accuracy']:.3f}")
print(f"composite MAE: {metrics['composite']['mae']:.2f}")
print(f"composite RMSE: {metrics['composite']['rmse']:.2f}")
print(f"composite R²: {metrics['composite']['r2']:+.3f}")

## 6. Download the trained model

The model is saved to `/content/Lucid/models/distilbert/`. Download it so you can commit + push, or upload to HuggingFace Hub.


In [ ]:
# Zip and download
!cd models && zip -r distilbert.zip distilbert/
from google.colab import files
files.download("models/distilbert.zip")

### Optional: push to HuggingFace Hub

Free hosting for the ~260MB checkpoint. The backend's `DeepPredictor` can then download on startup instead of bundling the weights.


In [ ]:
# Uncomment to push. Requires HF_TOKEN in environment.
# from huggingface_hub import HfApi, create_repo
# REPO_ID = "lindsaygross/lucid-distilbert"
# create_repo(REPO_ID, exist_ok=True, private=False)
# HfApi().upload_folder(
#     folder_path=str(OUTPUT_DIR),
#     repo_id=REPO_ID,
#     repo_type="model",
# )